In [2]:
import os
import json
import random
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
from bert_score import score
from dotenv import load_dotenv

In [7]:
# ----------------------------
# Configuration
# ----------------------------

SEED = 42
EVAL_JSONL_FILE = "../data/eval_1000_samples.jsonl"
EVAL_IMAGE_DIR = "../data/eval_images"

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

In [4]:
# ----------------------------
# Set random seeds
# ----------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [5]:
# ----------------------------
# Load LLaVA model (4-bit optimized)
# ----------------------------

def load_model():
    model_id = "llava-hf/llava-1.5-7b-hf"

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )

    processor = AutoProcessor.from_pretrained(model_id, use_fast=False, token=HF_TOKEN)
    model = LlavaForConditionalGeneration.from_pretrained(
        model_id,
        quantization_config=quant_config,
        device_map="auto",
        token=HF_TOKEN
    )

    return processor, model

processor, model = load_model()

Loading checkpoint shards: 100%|██████████| 3/3 [00:19<00:00,  6.38s/it]


In [9]:
# ----------------------------
# Load Evaluation JSONL
# ----------------------------

with open(EVAL_JSONL_FILE, "r") as f:
    eval_samples = [json.loads(line) for line in f]

print(f"Loaded {len(eval_samples)} evaluation samples.")

Loaded 1000 evaluation samples.


In [10]:
# ----------------------------
# Evaluate
# ----------------------------

predictions = []
references = []

print("Starting evaluation...")

for idx, example in enumerate(tqdm(eval_samples, desc="Evaluating")):
    
    # Get instruction and ground-truth response
    instruction = example["conversations"][0]["value"]
    gt_response = example["conversations"][-1]["value"]

    # Get image filename
    image_file = example.get("image", "").strip()
    if image_file.endswith(".JPG"):
        image_file = image_file[:-4] + ".jpg"

    img_path = os.path.join(EVAL_IMAGE_DIR, image_file)

    # If image does not exist, skip
    if not os.path.exists(img_path):
        print(f"WARNING: Image not found → {img_path}")
        continue

    # Load image
    image = Image.open(img_path).convert("RGB")

    # Prepare multimodal input
    inputs = processor(text=instruction, images=image, return_tensors="pt").to(model.device)

    # Generate response
    outputs = model.generate(**inputs, max_new_tokens=100)
    prediction = processor.decode(outputs[0], skip_special_tokens=True).strip()

    # Store prediction and reference
    predictions.append(prediction)
    references.append(gt_response)

# ----------------------------
# Calculate and show BERTScore
# ----------------------------

if len(predictions) == 0:
    print("No predictions to evaluate.")
else:
    print("Calculating BERTScore...")
    P, R, F1 = score(predictions, references, lang="en")
    print(f"\nBERTScore F1: {F1.mean().item():.4f}")


Starting evaluation...


Evaluating: 100%|██████████| 1000/1000 [1:08:53<00:00,  4.13s/it]


Calculating BERTScore...


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



BERTScore F1: 0.8693
